# Tugas 4 | Klasifikasi -> TFIDF & Word2Vec

klasifikasi menggunakan model *Logistic Regression*

## TF-IDF

- Ambil teks (clean_stemmed) → ubah jadi TF-IDF.

- Ambil label dari kolom Kategori.

- Split data jadi train & test. (20% test & 80% train)

- Latih model klasifikasi Logistic Regression

- Evaluasi akurasi.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Load dataset
df = pd.read_csv("detik_cleaned.csv")

# 2. Ambil teks dan label
texts = df["clean_stemmed"].dropna()
labels = df.loc[texts.index, "Kategori"]

# 3. TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(texts)

# 4. Split train-test
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, labels, test_size=0.2, random_state=42, stratify=labels
)

# 5. Latih model Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 6. Prediksi
y_pred = clf.predict(X_test)

# 7. Evaluasi
print("Akurasi:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred))


Akurasi: 0.95

Laporan Klasifikasi:
               precision    recall  f1-score   support

    Olahraga       1.00      0.90      0.95        10
  Pendidikan       0.91      1.00      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20



## Word Embedding

- Load dataset + Word2Vec

- Ambil rata-rata embedding tiap dokumen → matriks fitur.

- Ambil label (Kategori).

- Split train-test. (20% test & 80% train)

- Latih model klasifikasi Logistic Regression.

- Evaluasi akurasi + classification report.

In [2]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Load dataset
df = pd.read_csv("detik_cleaned.csv")
texts = df["clean_stemmed"].dropna()
labels = df.loc[texts.index, "Kategori"]

# 2. Tokenisasi
tokenized_texts = [t.split() for t in texts]

# 3. Latih Word2Vec (kalau belum ada model tersimpan)
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

# 4. Representasi dokumen = rata-rata embedding kata
doc_embeddings = []
for tokens in tokenized_texts:
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if vectors:
        doc_embeddings.append(np.mean(vectors, axis=0))
    else:
        doc_embeddings.append(np.zeros(100))  # kalau kosong

doc_embeddings = np.array(doc_embeddings)

# 5. Split train-test
X_train, X_test, y_train, y_test = train_test_split(
    doc_embeddings, labels, test_size=0.2, random_state=42, stratify=labels
)

# 6. Latih model Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 7. Prediksi
y_pred = clf.predict(X_test)

# 8. Evaluasi
print("Akurasi:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred))


Akurasi: 0.5

Laporan Klasifikasi:
               precision    recall  f1-score   support

    Olahraga       0.50      0.60      0.55        10
  Pendidikan       0.50      0.40      0.44        10

    accuracy                           0.50        20
   macro avg       0.50      0.50      0.49        20
weighted avg       0.50      0.50      0.49        20



## Kesimpulan

### Sifat Representasi TF-IDF vs Word2Vec

- TF-IDF → fokus pada kata yang benar-benar muncul di dokumen dan menekankan bobot kata yang unik/relevan dalam membedakan kelas (contoh: kata “gol”, “pemain”, “sekolah”, “guru”). Jadi sangat efektif di tugas klasifikasi teks karena langsung memanfaatkan frekuensi kata sebagai sinyal diskriminatif.

- Word2Vec → menghasilkan vektor kata yang mirip secara semantik, tapi ketika dirata-ratakan, informasi diskriminatif antar kategori bisa hilang. Misalnya kata "guru" dan "pelatih" bisa dekat di embedding space, padahal untuk klasifikasi "pendidikan" vs "olahraga", justru kata itu penting dibedakan.

### Ukuran dataset berpengaruh

- Kalau dataset kecil, TF-IDF biasanya lebih unggul karena tidak butuh banyak data untuk belajar.

- Word2Vec butuh corpus besar supaya embedding kata benar-benar bermakna. Kalau hanya dilatih di dataset kecil (seperti detik_cleaned.csv punyamu), kualitas vektornya kurang bagus → performa turun.

### Rata-rata Embedding Menghilangkan Struktur Konteks

- Word2Vec saya menghitung dokumen embedding dengan mean pooling (np.mean(vectors, axis=0)).

- Ini menyebabkan urutan kata, intensitas kata, bahkan perbedaan halus antar kata hilang. Jadi dokumen dari dua kategori bisa terlihat mirip kalau kata-katanya punya embedding yang dekat.

- TF-IDF tetap mempertahankan perbedaan kata per kategori, sehingga classifier lebih mudah belajar.

Simpel nya TF-IDF menang di sini karena dataset kecil dan modelnya linear (Logistic Regression).

Word2Vec kalah karena:

- Dilatih dari nol di dataset kecil → embedding kurang bagus

- Representasi rata-rata embedding kehilangan informasi kata penting

Kalau ingin Word2Vec lebih baik, ada beberapa trik:

- Gunakan pre-trained Word2Vec Indonesia (misalnya dari FastText, IndoWord2Vec).

- Coba representasi selain mean pooling → bisa pakai TF-IDF weighted average embedding.

- Kalau dataset besar, Word2Vec mulai menunjukkan keunggulan.